# Olist Commerce Intelligence

This notebook presents the core analytical story. The production-style transformations live in `src/`, so the notebook stays readable and reproducible.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import plotly.express as px

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
orders = pd.read_parquet(ROOT / 'data/processed/orders_enriched.parquet')
delivered = orders.query("order_status == 'delivered'").copy()
metrics = json.loads((ROOT / 'reports/model_metrics.json').read_text())
orders.shape, delivered.shape

## 1. Executive KPIs

Revenue uses the total payment captured for delivered orders. Late delivery means the recorded delivery timestamp is after the promised date.

In [ ]:
pd.Series({
    'orders': orders.order_id.nunique(),
    'delivered_orders': delivered.order_id.nunique(),
    'revenue_brl': delivered.total_payment.sum(),
    'average_order_value_brl': delivered.total_payment.mean(),
    'late_rate': delivered.is_late.mean(),
    'average_review': delivered.review_score.mean(),
})

In [ ]:
monthly = delivered.groupby('purchase_year_month', as_index=False).agg(
    revenue=('total_payment', 'sum'), orders=('order_id', 'nunique'), late_rate=('is_late', 'mean')
)
px.line(monthly, x='purchase_year_month', y='revenue', markers=True, title='Monthly delivered revenue')

## 2. Delivery experience

The comparison below is observational. A difference in reviews does not by itself prove that a delay caused the rating.

In [ ]:
review_by_delivery = delivered.groupby('is_late', as_index=False).agg(
    orders=('order_id', 'nunique'), average_review=('review_score', 'mean')
)
review_by_delivery['delivery_result'] = review_by_delivery.is_late.map({0.0: 'On time', 1.0: 'Late'})
review_by_delivery

In [ ]:
category = delivered.groupby('primary_category', as_index=False).agg(
    orders=('order_id', 'nunique'), revenue=('total_payment', 'sum'), late_rate=('is_late', 'mean')
).query('orders >= 500')
px.scatter(category, x='orders', y='late_rate', size='revenue', hover_name='primary_category',
           title='Category scale and delivery risk')

## 3. Customer behavior

RFM segments are included as descriptive prioritization. Repeat purchase is rare in the observation window, which limits the strength of frequency-based conclusions.

In [ ]:
rfm = pd.read_csv(ROOT / 'data/processed/rfm_segments.csv')
rfm.groupby('segment', as_index=False).agg(customers=('customer_unique_id', 'nunique'), revenue=('monetary', 'sum'))

## 4. Predictive model

The model uses a chronological holdout and only checkout-time fields. PR-AUC is the primary metric because late deliveries are uncommon.

In [ ]:
pd.DataFrame([metrics['baseline_test'], metrics['test']], index=['Dummy baseline', metrics['selected_model']])

In [ ]:
importance = pd.read_csv(ROOT / 'data/processed/feature_importance.csv').head(15).sort_values('importance')
px.bar(importance, x='importance', y='feature', orientation='h', title='Strongest model signals')

## Decision

Use the score to rank orders for proactive monitoring, then test an intervention. Do not interpret feature coefficients as causal effects or deploy the model without validation on current operational data.